# 🧪 SwinIR + EasyOCR Ablation Study (Kaggle)

This notebook evaluates the impact of **Super-Resolution (SwinIR)** on **EasyOCR + TrOCR** performance.

## Instructions
1. **Add Data**: Upload dataset containing `textvqa_dataset.json` and images.
2. **Add Models**: Upload `best_models_for_testing` containing SwinIR and TrOCR LoRA.
3. **Run All**: Compares Pretrained vs Fine-tuned SwinIR.

In [ ]:
# 📦 Install Dependencies
!pip install -q easyocr transformers peft rapidfuzz jiwer opencv-python-headless qwen_vl_utils "numpy<2.0"

In [ ]:
# 🕵️‍♂️ ROBUST CONFIGURATION (GLOB SEARCH)
import glob
import os
import sys
from pathlib import Path

print("🔍 Searching for 'textvqa_dataset.json' using GLOB...")
json_candidates = glob.glob("/kaggle/input/**/textvqa_dataset.json", recursive=True)

if json_candidates:
    ANNO_PATH = Path(json_candidates[0])
    print(f"✅ FOUND Annotation File: {ANNO_PATH}")
    
    # Heuristic to find image folder
    BASE_DIR = ANNO_PATH.parent
    if (BASE_DIR / "DATA").exists():
        DATA_PATH = BASE_DIR / "DATA"
    else:
        DATA_PATH = BASE_DIR
    print(f"   -> Base Image Path set to: {DATA_PATH}")
else:
    print("❌ ERROR: Could not find 'textvqa_dataset.json' anywhere.")
    ANNO_PATH = None
    DATA_PATH = Path("/kaggle/input")

# Find SwinIR Arch
swin_candidates = glob.glob("/kaggle/input/**/swinir_arch.py", recursive=True)
if swin_candidates:
    MODEL_BASE_PATH = Path(swin_candidates[0]).parent
    sys.path.append(str(MODEL_BASE_PATH))
    print(f"✅ FOUND Models at: {MODEL_BASE_PATH}")
else:
    MODEL_BASE_PATH = Path("ERROR_MODELS")
    print("❌ ERROR: Could not find 'swinir_arch.py'.")

# Set Paths
PRETRAINED_SWINIR_PATH = MODEL_BASE_PATH / "swinir_medium_pretrained_bsrgan.pth"
FINETUNED_SWINIR_PATH = MODEL_BASE_PATH / "swinir_medium_textzoom_curriculum.pth"
LORA_PATH = MODEL_BASE_PATH / "trocr_base_lora_textzoom"

# Check Lora
if not LORA_PATH.exists():
    l_cand = glob.glob("/kaggle/input/**/trocr_base_lora_textzoom/adapter_config.json", recursive=True)
    if l_cand:
        LORA_PATH = Path(l_cand[0]).parent
        print(f"   -> Found LoRA at: {LORA_PATH}")

In [ ]:
import torch
import cv2
import easyocr
import numpy as np
from PIL import Image
from tqdm import tqdm
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from peft import PeftModel
import json

# Import SwinIR
try:
    from swinir_arch import SwinIR
except ImportError:
    pass

# 🧠 LOAD MODELS

def load_swinir(model_path):
    print(f"Loading SwinIR from {model_path}...")
    args = {
        'upscale': 4,
        'in_chans': 3,
        'img_size': 64,
        'window_size': 8,
        'img_range': 1.0,
        'depths': [6, 6, 6, 6, 6, 6],
        'embed_dim': 180,
        'num_heads': [6, 6, 6, 6, 6, 6],
        'mlp_ratio': 2.0,
        'upsampler': 'nearest+conv', 
        'resi_connection': '1conv'
    }
    model = SwinIR(**args)
    checkpoint = torch.load(model_path, map_location='cuda')
    if 'params_ema' in checkpoint:
        state_dict = checkpoint['params_ema']
    elif 'params' in checkpoint:
        state_dict = checkpoint['params']
    else:
        state_dict = checkpoint
    model.load_state_dict(state_dict, strict=True)
    model.cuda().eval()
    return model

def load_easyocr_pipeline():
    print("Loading EasyOCR + TrOCR...")
    reader = easyocr.Reader(['en'], gpu=True, verbose=False)
    
    proc = TrOCRProcessor.from_pretrained("microsoft/trocr-base-str")
    base = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-str").to("cuda")
    
    if LORA_PATH and LORA_PATH.exists():
        try:
            model = PeftModel.from_pretrained(base, str(LORA_PATH))
            model = model.merge_and_unload()
            print("   -> Loaded TrOCR LoRA")
        except: 
            model = base
    else:
        model = base
        print("   -> Using Base TrOCR")
    model.eval()
    return reader, model, proc

In [ ]:
# 🔄 PIPELINE UTILS

def swinir_inference(model, img_path):
    img_path = str(img_path)
    img_lq = cv2.imread(img_path, cv2.IMREAD_COLOR).astype(np.float32) / 255.
    img_lq = np.transpose(img_lq if img_lq.shape[2] == 3 else img_lq[:, :, [2, 1, 0]], (2, 0, 1)) 
    img_lq = torch.from_numpy(img_lq).float().unsqueeze(0).cuda()
    
    # Pad
    window_size = 8
    _, _, h_old, w_old = img_lq.size()
    h_pad = (h_old // window_size + 1) * window_size - h_old
    w_pad = (w_old // window_size + 1) * window_size - w_old
    img_lq = torch.cat([img_lq, torch.flip(img_lq, [2])], 2)[:, :, :h_old + h_pad, :]
    img_lq = torch.cat([img_lq, torch.flip(img_lq, [3])], 3)[:, :, :, :w_old + w_pad]

    with torch.no_grad():
        output = model(img_lq)
        output = output[..., :h_old * 4, :w_old * 4]

    output = output.data.squeeze().float().cpu().clamp_(0, 1).numpy()
    if output.ndim == 3:
        output = np.transpose(output[[2, 1, 0], :, :], (1, 2, 0)) 
    output = (output * 255.0).round().astype(np.uint8)
    return output

def ocr_pipeline(reader, trocr_model, trocr_proc, img_bgr):
    # 1. Det (EasyOCR)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    res = reader.readtext(img_rgb)
    
    boxes = []
    for (box, _, _) in res:
        pts = np.array(box).astype(int)
        boxes.append((min(pts[:,0]), min(pts[:,1]), max(pts[:,0]), max(pts[:,1])))
    boxes.sort(key=lambda b: (b[1], b[0]))
    
    # 2. Rec (TrOCR)
    full_text = []
    pil_img = Image.fromarray(img_rgb)
    for xmin, ymin, xmax, ymax in boxes:
        crop = pil_img.crop((xmin, ymin, xmax, ymax))
        if crop.size[0]<2 or crop.size[1]<2: continue
        pixel_values = trocr_proc(images=crop, return_tensors="pt").pixel_values.to("cuda")
        with torch.no_grad():
            ids = trocr_model.generate(pixel_values, max_new_tokens=20)
            text = trocr_proc.batch_decode(ids, skip_special_tokens=True)[0]
        if text.strip(): full_text.append(text.strip())
    return " ".join(full_text)

def calculate_anls(pred, gt_list):
    from rapidfuzz import fuzz
    if not pred and not gt_list: return 1.0
    best = 0.0
    for gt in gt_list:
        sim = fuzz.ratio(pred.lower(), gt.lower())/100.0
        if sim < 0.5: sim = 0.0
        best = max(best, sim)
    return best

In [ ]:
# 🚀 MAIN EVALUATION (NO PADDLE)

def run_ablation(name, swin_model, reader, tm, tp, samples):
    print(f"\n📢 START: {name}")
    scores = []
    missing = 0
    
    for i, sample in enumerate(tqdm(samples)):
        img_id = sample['image_id']
        # Robust Find Strategy
        p = None
        cands = [
            DATA_PATH/f"{img_id}.jpg", 
            DATA_PATH/f"{img_id}.png", 
            DATA_PATH.parent/f"{img_id}.jpg",
            DATA_PATH.parent/"DATA"/f"{img_id}.jpg"
        ]
        for c in cands: 
            if c.exists(): 
                p=c; break
        
        # Debug first miss
        if not p:
            if missing == 0:
                print(f"\n⚠️ [DEBUG] First Missing Image: {img_id}")
            missing += 1
            continue
            
        try:
            # 1. SwinIR
            sr_bgr = swinir_inference(swin_model, p)
            
            # 2. OCR
            text = ocr_pipeline(reader, tm, tp, sr_bgr)
            
            # 3. Score
            score = calculate_anls(text, sample.get('answers', []))
            scores.append(score)
            
            if (i+1)%50==0: print(f"[{i+1}] Mean: {np.mean(scores):.4f}")
        except Exception as e:
            print(f"Err {img_id}: {e}")

    mean_score = np.mean(scores) if scores else 0
    print(f"🏆 {name} RESULT: {mean_score:.4f} (Skipped: {missing})")
    return mean_score

# -- RUN --
if ANNO_PATH and ANNO_PATH.exists():
    with open(ANNO_PATH) as f: 
        samples = json.load(f)['data']

    # Load Tools
    reader, tm, tp = load_easyocr_pipeline()
    swin_pt = load_swinir(PRETRAINED_SWINIR_PATH)
    swin_ft = load_swinir(FINETUNED_SWINIR_PATH)

    s1 = run_ablation("Pretrained SwinIR", swin_pt, reader, tm, tp, samples)
    s2 = run_ablation("Finetuned SwinIR", swin_ft, reader, tm, tp, samples)
    
    print(f"\nFINAL COMPARISON:\nPretrained: {s1:.4f}\nFinetuned:  {s2:.4f}")
else:
    print(f"❌ CRITICAL: Annotation file not found. Check upload.")